# MNIST — DDPM Diffusion (Cosine Schedule, DDIM & DPM-Solver++ 2M)

Implements **Section 12.2** (`mnist_diffusion.ipynb`) of the Generative Models Benchmark Specification.
All six model notebooks share the exact same 10-step structure; only `DATASET`, `MODEL_NAME`, and the Section 9 model-specific tuning alternatives differ.

In [ ]:
# 1. Colab setup cell (identical in every notebook per Section 3 & Section 12)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/alisamadian/generative-models-benchmark.git'
REPO_DIR = Path('/content/generative-models-benchmark')
DRIVE_REPO_DIR = Path('/content/drive/MyDrive/generative-models-benchmark')

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    if DRIVE_REPO_DIR.exists() and (DRIVE_REPO_DIR / 'common').exists():
        os.chdir(DRIVE_REPO_DIR)
    else:
        if not REPO_DIR.exists():
            subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
        else:
            subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=False)
        os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    # Local execution: ensure working directory is the repository root
    if Path.cwd().name == 'notebooks':
        os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Reload common.* modules if already imported in this Python kernel after git pull
import importlib
for _mod_name in list(sys.modules.keys()):
    if _mod_name == 'common' or _mod_name.startswith('common.'):
        importlib.reload(sys.modules[_mod_name])

import torch
from common.config import SPLIT_SEED, protocol_dict, protocol_hash
from common.utils import library_versions, mount_drive, set_seed

DRIVE_PATHS = mount_drive()
set_seed(SPLIT_SEED)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {gpu_name} ({gpu_mem_gb:.2f} GB VRAM)')
else:
    print('GPU: None (running on CPU)')

print('Working directory:', Path.cwd())
print('Library versions:', library_versions())
print('Drive/Cache paths:', {k: str(v) for k, v in DRIVE_PATHS.items()})

## 2. Configuration, Protocol Hash & Mode (Section 5, Section 8, Section 12.2)

In [ ]:
DATASET = 'mnist'
MODEL_NAME = 'diffusion'
MODE = 'final'  # Options: 'pilot' | 'tune' | 'final'
SEED = 0        # Training replicate seed: 0 or 1 (Section 5)

import json
from common.config import (
    BATCH_SIZE, CKPT_INTERVAL, EMA_DECAY, SAMPLE_GRID_COUNT, SAMPLE_GRID_SEED,
    TRAIN_STEPS, TUNE_FRACTION, TUNE_SEED, protocol_dict, protocol_hash,
)

print(f'=== {DATASET.upper()} | {MODEL_NAME.upper()} | MODE={MODE} | SEED={SEED} ===')
print('Protocol SHA-256 Hash:', protocol_hash())
print('Dataset Train Steps:', TRAIN_STEPS[DATASET], '| Checkpoint Interval:', CKPT_INTERVAL[DATASET])

## 3. Load Saved Splits & Run Data Validation Report (Train & Val Only; Test Locked)

In [ ]:
from common.data import get_split_tensors, load_raw, load_splits, pad_mnist, to_model_range, validate_and_report
from common.preflight import check_p13_test_access_guard

raw_data = load_raw(DATASET, data_root='./data')
split_data = load_splits(DATASET, split_path=f'./splits/{DATASET}_split.json')

# Verify test split is strictly locked outside evaluate_final() (Check P13)
assert check_p13_test_access_guard()['passed']

data_card = validate_and_report(
    dataset=DATASET,
    raw_data=raw_data,
    split_data=split_data,
    include_test=False,
    show_plots=True,
)

## 4. Build Model, Print Architecture Summary & Verify Parameter Count (Section 7)

In [ ]:
from common.nets import build_model, count_parameters
from common.runner import default_hparams

init_hparams = default_hparams(MODEL_NAME)
model_preview = build_model(DATASET, MODEL_NAME, seed=SEED, hparams=init_hparams)
unet_ref = build_model(DATASET, 'diffusion', seed=SEED)

param_count = count_parameters(model_preview)
unet_param_count = count_parameters(unet_ref)
rel_diff = (param_count - unet_param_count) / float(unet_param_count)

print(f'Model Class: {model_preview.__class__.__name__}')
print(f'Trainable Parameters: {param_count:,} (Reference U-Net: {unet_param_count:,}, diff: {rel_diff:+.1%})')
assert abs(rel_diff) <= 0.25, f'Parameter count {param_count:,} exceeds 25% of U-Net {unet_param_count:,}!'
del model_preview, unet_ref

## 5. Pilot Mode: Speed, Peak GPU Memory & Tiny-Batch Overfit Check (Section 8, Checks P5 & P14)

In [ ]:
from common.preflight import check_p5_tiny_batch_overfit, check_p14_memory_and_speed

if MODE == 'pilot':
    p14_res = check_p14_memory_and_speed(dataset=DATASET, steps=200, batch_size=BATCH_SIZE)
    x_tr_u8, _ = get_split_tensors(DATASET, 'train', raw_data=raw_data, split_data=split_data)
    x16 = x_tr_u8[:16]
    if DATASET == 'mnist':
        x16 = pad_mnist(x16)
    x16_flt = to_model_range(x16)
    p5_res = check_p5_tiny_batch_overfit(dataset=DATASET, steps=500, tiny_batch=x16_flt)
    print('Pilot P14 (Speed & Memory):', p14_res)
    print('Pilot P5 (16-Image Overfit):', p5_res)

## 6. Tune Mode: Run 4 Pre-Committed Validation Trials & Freeze Winner (Section 9)

In [ ]:
from common.metrics import evaluate_block, reference_stats
from common.runner import freeze_chosen_hparams, get_tuning_trials, train_model
from common.utils import write_results_json

if MODE == 'tune':
    ref_stats = reference_stats(
        dataset=DATASET,
        cache_dir=DRIVE_PATHS['fid_stats'],
        raw_data=raw_data,
        split_data=split_data,
        include_val=True,
    )
    tune_steps = int(round(TUNE_FRACTION * TRAIN_STEPS[DATASET]))
    
    # Run Trials 1 to 3 (LR sweep in {1e-4, 2e-4, 4e-4} with default configuration)
    trials_1_to_3 = get_tuning_trials(MODEL_NAME)[:3]
    scores_1_to_3 = []
    
    for trial_hp in trials_1_to_3:
        t_idx = int(trial_hp['trial_index'])
        ckpt_dir = DRIVE_PATHS['checkpoints'] / DATASET / MODEL_NAME / f'tune_trial{t_idx}'
        csv_path = DRIVE_PATHS['logs'] / f'{DATASET}_{MODEL_NAME}_tune_trial{t_idx}.csv'
        tr_out = train_model(
            dataset=DATASET,
            model_name=MODEL_NAME,
            seed=TUNE_SEED,
            hparams=trial_hp,
            max_steps=tune_steps,
            ckpt_dir=ckpt_dir,
            csv_log_path=csv_path,
            ref_stats_dict=ref_stats,
            raw_data=raw_data,
            split_data=split_data,
            monitor_points=[tune_steps],
        )
        val_fid = tr_out['val_fid_history'][-1]['val_fid']
        val_loss = tr_out['val_loss_history'][-1]['val_loss']
        scores_1_to_3.append((val_fid, float(trial_hp['lr'])))
        write_results_json(
            Path('results/tuning') / f'{DATASET}_{MODEL_NAME}_trial{t_idx}.json',
            {
                'run_id': f'{DATASET}_{MODEL_NAME}_tune_trial{t_idx}',
                'dataset': DATASET,
                'model': MODEL_NAME,
                'mode': 'tune',
                'seed': TUNE_SEED,
                'trial_index': t_idx,
                'hparams': {k: v for k, v in trial_hp.items() if k != 'trial_index'},
                'train_steps': tr_out['final_step'],
                'val_fid': val_fid,
                'val_loss': val_loss,
                'test_touched': False,
            },
            overwrite_non_final=True,
        )
    
    # Trial 4: Best learning rate from Trials 1-3 + method-specific alternative
    best_lr = min(scores_1_to_3, key=lambda x: x[0])[1]
    trial_4_hp = get_tuning_trials(MODEL_NAME, best_lr_from_1_to_3=best_lr)[3]
    ckpt_dir_4 = DRIVE_PATHS['checkpoints'] / DATASET / MODEL_NAME / 'tune_trial4'
    csv_path_4 = DRIVE_PATHS['logs'] / f'{DATASET}_{MODEL_NAME}_tune_trial4.csv'
    tr_out_4 = train_model(
        dataset=DATASET,
        model_name=MODEL_NAME,
        seed=TUNE_SEED,
        hparams=trial_4_hp,
        max_steps=tune_steps,
        ckpt_dir=ckpt_dir_4,
        csv_log_path=csv_path_4,
        ref_stats_dict=ref_stats,
        raw_data=raw_data,
        split_data=split_data,
        monitor_points=[tune_steps],
    )
    write_results_json(
        Path('results/tuning') / f'{DATASET}_{MODEL_NAME}_trial4.json',
        {
            'run_id': f'{DATASET}_{MODEL_NAME}_tune_trial4',
            'dataset': DATASET,
            'model': MODEL_NAME,
            'mode': 'tune',
            'seed': TUNE_SEED,
            'trial_index': 4,
            'hparams': {k: v for k, v in trial_4_hp.items() if k != 'trial_index'},
            'train_steps': tr_out_4['final_step'],
            'val_fid': tr_out_4['val_fid_history'][-1]['val_fid'],
            'val_loss': tr_out_4['val_loss_history'][-1]['val_loss'],
            'test_touched': False,
        },
        overwrite_non_final=True,
    )
    
    # Freeze winner to results/chosen/{dataset}_{model}.json
    chosen_info = freeze_chosen_hparams(DATASET, MODEL_NAME)
    print('Frozen Chosen Hyperparameters:', json.dumps(chosen_info, indent=2))

## 7. Final Mode: Verify Frozen Hyperparameters & Train with Automatic Resume (Section 8, Section 12.2)

In [ ]:
from common.metrics import reference_stats
from common.runner import load_chosen_hparams, train_model

train_result = None
if MODE == 'final':
    # Refuses to start if results/chosen/{dataset}_{model}.json is missing (Section 6.2 Rule 8)
    chosen_record = load_chosen_hparams(DATASET, MODEL_NAME)
    frozen_hparams = chosen_record['hparams']
    print('Loaded Frozen Hyperparameters:', frozen_hparams)
    
    ref_stats = reference_stats(
        dataset=DATASET,
        cache_dir=DRIVE_PATHS['fid_stats'],
        raw_data=raw_data,
        split_data=split_data,
        include_val=True,
    )
    ckpt_dir = DRIVE_PATHS['checkpoints'] / DATASET / MODEL_NAME / f'seed{SEED}'
    csv_path = DRIVE_PATHS['logs'] / f'{DATASET}_{MODEL_NAME}_seed{SEED}_final.csv'
    
    train_result = train_model(
        dataset=DATASET,
        model_name=MODEL_NAME,
        seed=SEED,
        hparams=frozen_hparams,
        max_steps=TRAIN_STEPS[DATASET],
        ckpt_dir=ckpt_dir,
        csv_log_path=csv_path,
        ref_stats_dict=ref_stats,
        raw_data=raw_data,
        split_data=split_data,
    )

## 8. Plot Training Loss, Validation Loss & Validation FID vs Steps (Section 12.2 Step 8)

In [ ]:
from common.runner import plot_training_curves

if train_result is not None:
    curve_path = Path('figures') / f'curves_{DATASET}_{MODEL_NAME}_seed{SEED}.png'
    plot_training_curves(
        train_result,
        dataset=DATASET,
        model_name=MODEL_NAME,
        seed=SEED,
        save_path=curve_path,
        show_plot=True,
    )

## 9. Generate Fixed-Seed Sample Grid (seed=777, n=64) & Run Check V13 (Section 10.3, 12.2 Step 9)

In [ ]:
from common.data import validate_generated_samples
from common.samplers import generate

if train_result is not None:
    ema_model = train_result['ema_model']
    sched = str(frozen_hparams.get('schedule', 'cosine'))
    first_sampler = 'vae' if MODEL_NAME == 'vae' else ('ddim' if MODEL_NAME == 'diffusion' else 'euler')
    nfes_for_grid = [1] if MODEL_NAME == 'vae' else [20, 100]
    
    for nfe_g in nfes_for_grid:
        grid_samples = generate(
            ema_model,
            sampler=first_sampler,
            nfe=nfe_g,
            n=SAMPLE_GRID_COUNT,
            seed=SAMPLE_GRID_SEED,
            dataset=DATASET,
            schedule=sched,
        )
        grid_fig = Path('figures') / f'f4_grid_{DATASET}_{MODEL_NAME}_seed{SEED}_nfe{nfe_g}.png'
        validate_generated_samples(
            grid_samples,
            dataset=DATASET,
            expected_count=SAMPLE_GRID_COUNT,
            show_plot=True,
            save_fig_path=grid_fig,
            title=f'F4: {DATASET.upper()} {MODEL_NAME} (seed={SEED}, sampler={first_sampler}, NFE={nfe_g})',
        )

## 10. Final Mode Only: Call `evaluate_final()` Once (Blocks B–H) & Write Results JSON (Section 10.2, Appendix A)

In [ ]:
from common.metrics import evaluate_final, train_eval_classifier
from common.nets import count_parameters
from common.utils import write_results_json

if MODE == 'final' and train_result is not None:
    clf_ckpt = DRIVE_PATHS['checkpoints'] / DATASET / 'classifier' / 'best_classifier.pt'
    classifier, _ = train_eval_classifier(
        dataset=DATASET,
        save_path=clf_ckpt,
        raw_data=raw_data,
        split_data=split_data,
    )
    metrics_bundle = evaluate_final(
        ema_model=train_result['ema_model'],
        dataset=DATASET,
        model_name=MODEL_NAME,
        seed=SEED,
        ref_stats_dict=ref_stats,
        classifier=classifier,
        schedule=str(frozen_hparams.get('schedule', 'cosine')),
        train_steps=train_result['final_step'],
        train_seconds=train_result['train_seconds'],
        it_per_sec=train_result['it_per_sec'],
        raw_data=raw_data,
        split_data=split_data,
        figures_dir='./figures',
    )
    
    # Build Appendix A compliant write-once JSON payload
    final_payload = {
        'run_id': f'{DATASET}_{MODEL_NAME}_seed{SEED}_final',
        'dataset': DATASET,
        'model': MODEL_NAME,
        'mode': 'final',
        'seed': SEED,
        'protocol_hash': protocol_hash(),
        'config_hash': train_result['config_hash'],
        'split_hash': split_data['split_hash'],
        'hparams': frozen_hparams,
        'parameters': count_parameters(train_result['ema_model'], trainable_only=False),
        'train_steps': train_result['final_step'],
        'train_seconds': round(train_result['train_seconds'], 3),
        'it_per_sec': round(train_result['it_per_sec'], 4),
        'ema_decay': EMA_DECAY[DATASET],
        'checkpoint_path': train_result['checkpoint_path'],
        'val_fid_history': train_result['val_fid_history'],
        'data_card': data_card,
        'metrics': metrics_bundle,
        'test_touched': True,
        'rerun_reason': None,
    }
    out_json = Path('results/final') / f'{DATASET}_{MODEL_NAME}_seed{SEED}.json'
    write_results_json(out_json, final_payload, overwrite_non_final=False)
    print(f'Saved write-once final results JSON to {out_json}')